# Synchronization Primitives — Experiment Analysis

In [7]:
import subprocess
from pathlib import Path

import pandas as pd
import plotly.express as px

PYTHON_VERSIONS = ["3.13", "3.13t", "3.14", "3.14t"]

def run_benchmark(experiment: str, pythons: list = PYTHON_VERSIONS) -> pd.DataFrame:
    subprocess.run(
        ["nox", "-s", "experiments", "-p", *pythons, "--", experiment],
        check=True,
    )

    frames = []
    for python in pythons:
        path = Path(f"results/{experiment}_{python}.json")
        df = pd.read_json(path)
        df["python"] = python
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

In [9]:
df01 = run_benchmark("race_condition", pythons=["3.13"])
df01

nox > Running session experiments-3.13
nox > Creating virtual environment (uv) using python3.13 in .nox/experiments-3-13
nox > uv pip install pandas plotly
nox > python experiments/race_condition.py --output results/race_condition_3.13.json
nox > Session experiments-3.13 was successful in 35 seconds.


,python_version,case,tasks,result,expected,time_s,python
0,3.13.3,threading / unsafe,3,500,1500,0.628,3.13
1,3.13.3,threading / unsafe,2,500,1000,0.633,3.13
2,3.13.3,threading / unsafe,5,500,2500,0.635,3.13
3,3.13.3,threading / unsafe,4,500,2000,0.642,3.13
4,3.13.3,processes / unsafe,2,500,1000,0.954,3.13
5,3.13.3,processes / unsafe,3,500,1500,0.963,3.13
6,3.13.3,processes / unsafe,4,501,2000,0.976,3.13
7,3.13.3,processes / unsafe,5,501,2500,0.980,3.13
8,3.13.3,threading / safe,2,1000,1000,1.265,3.13
9,3.13.3,processes / safe,2,1000,1000,1.560,3.13


## 01. Race Condition

In [23]:
CASE_COLORS = {
    "threading / unsafe": "#93C4F9",  # light blue
    "threading / safe  ": "#1A6BC1",  # dark blue
    "processes / unsafe": "#F4B07A",  # light orange
    "processes / safe  ": "#C45A00",  # dark orange
}

px.bar(
    df01,
    x="tasks",
    y="time_s",
    color="case",
    color_discrete_map=CASE_COLORS,
    barmode="group",
    title="Execution time by concurrency level",
    labels={"time_s": "time (s)", "tasks": "concurrent tasks"},
)

## 02. Deadlock

In [1]:
import pandas as pd
import plotly.express as px
from experiments.deadlock import demo_deadlock, demo_fixed, demo_deadlock_3, demo_fixed_3

In [2]:
df_deadlock = demo_deadlock()
df_fixed = demo_fixed()

print("--- deadlock ---")
print(df_deadlock.to_string(index=False))
print("\n--- fixed (lock ordering) ---")
print(df_fixed.to_string(index=False))

--- deadlock ---
thread      event  time_s
    T1 acquired A   0.000
    T2 acquired B   0.001
    T1 DEADLOCKED   2.011
    T2 DEADLOCKED   2.011

--- fixed (lock ordering) ---
thread      event  time_s
    T1 acquired A   0.000
    T1 acquired B   0.505
    T1  completed   1.010
    T2 acquired A   1.010
    T2 acquired B   1.515
    T2  completed   2.020


In [4]:
EVENT_COLORS = {
    "acquired A":  "#1A6BC1",
    "acquired B":  "#93C4F9",
    "completed":   "#2CA02C",
    "DEADLOCKED":  "#D62728",
}

px.scatter(
    df_deadlock,
    x="time_s",
    y="thread",
    color="event",
    color_discrete_map=EVENT_COLORS,
    text="event",
    title="Lock acquisition timeline — deadlock scenario",
    labels={"time_s": "time (s)", "thread": "thread"},
)

In [5]:
px.scatter(
    df_fixed,
    x="time_s",
    y="thread",
    color="event",
    color_discrete_map=EVENT_COLORS,
    text="event",
    title="Lock acquisition timeline — fixed (lock ordering)",
    labels={"time_s": "time (s)", "thread": "thread"},
)

In [3]:
df_deadlock_3 = demo_deadlock_3()
df_fixed_3 = demo_fixed_3()

print("--- deadlock (3 threads) ---")
print(df_deadlock_3.to_string(index=False))
print("\n--- fixed (3 threads) ---")
print(df_fixed_3.to_string(index=False))

--- deadlock (3 threads) ---
thread      event  time_s
    T1 acquired A   0.001
    T2 acquired B   0.001
    T3 acquired C   0.001
    T1 DEADLOCKED   3.016
    T2 DEADLOCKED   3.016
    T3 DEADLOCKED   3.016

--- fixed (3 threads) ---
thread      event  time_s
    T1 acquired A   0.000
    T2 acquired B   0.000
    T2 acquired C   0.501
    T2  completed   1.006
    T1 acquired B   1.006
    T1  completed   1.510
    T3 acquired A   1.510
    T3 acquired C   2.015
    T3  completed   2.521


In [4]:
EVENT_COLORS_3 = {
    "acquired A":  "#1A6BC1",
    "acquired B":  "#93C4F9",
    "acquired C":  "#BDD7EE",
    "completed":   "#2CA02C",
    "DEADLOCKED":  "#D62728",
}

px.scatter(
    df_deadlock_3,
    x="time_s",
    y="thread",
    color="event",
    color_discrete_map=EVENT_COLORS_3,
    text="event",
    title="Lock acquisition timeline — 3-thread deadlock (A→B, B→C, C→A)",
    labels={"time_s": "time (s)", "thread": "thread"},
)

In [5]:
px.scatter(
    df_fixed_3,
    x="time_s",
    y="thread",
    color="event",
    color_discrete_map=EVENT_COLORS_3,
    text="event",
    title="Lock acquisition timeline — 3-thread fixed (global ordering A < B < C)",
    labels={"time_s": "time (s)", "thread": "thread"},
)

## 03. strace — uncontended vs contended

In [ ]:
from experiments.strace_lock import uncontended, contended

## 04. GIL Benchmark

In [ ]:
from experiments.gil_benchmark import run_single, run_threaded

## 05. Lock Levels

In [ ]:
from experiments.lock_levels import bench_threading_lock, bench_asyncio_lock, bench_multiprocessing_lock